# LeetCode 1474 — Delete N Nodes After M Nodes of a Linked List bằng Python

**Link đề:** https://leetcode.com/problems/delete-n-nodes-after-m-nodes-of-a-linked-list/
**Độ khó:** Easy · **Chủ đề:** Linked List

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def deleteNodes(self, head: Optional[ListNode], m: int, n: int) -> Optional[ListNode]:
        ...
```

### Mục tiêu học
1. Tự định nghĩa `ListNode`, hàm **dựng** danh sách liên kết từ list Python và **chuyển ngược** để in/kiểm tra.
2. Hiểu thao tác cốt lõi của linked list: **bỏ qua nút** bằng cách nối lại con trỏ `prev.next = ...`.
3. Đi qua 5 lời giải: chuyển sang list rồi lọc theo chỉ số → tạo danh sách mới → **in-place hai pha "giữ m, bỏ n"** → in-place một vòng với bộ đếm `% (m+n)` → đệ quy.
4. Trace con trỏ từng bước, chứng minh invariant, phân tích Time/Space (follow-up: sửa **tại chỗ**).
5. Biến thể: *Remove Linked List Elements*, xóa mỗi nút thứ `k`, *Remove Nth Node From End*.

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho `head` của một danh sách liên kết đơn và hai số nguyên `m`, `n`. Duyệt danh sách và xóa nút theo quy tắc:
1. Bắt đầu với `head` là nút hiện tại.
2. **Giữ** `m` nút đầu tiên tính từ nút hiện tại.
3. **Xóa** `n` nút kế tiếp.
4. Lặp lại bước 2 và 3 cho tới khi hết danh sách.

Trả về `head` của danh sách sau khi xóa.

| Ví dụ | `head` | `m` | `n` | Output |
|---|---|---:|---:|---|
| 1 | `[1,2,3,4,5,6,7,8,9,10,11,12,13]` | 2 | 3 | `[1,2,6,7,11,12]` |
| 2 | `[1,2,3,4,5,6,7,8,9,10,11]` | 1 | 3 | `[1,5,9]` |

Ví dụ 1: giữ `1,2` – xóa `3,4,5` – giữ `6,7` – xóa `8,9,10` – giữ `11,12` – xóa `13` (chỉ còn 1 nút để xóa thì xóa hết phần còn lại).

**Constraints** (theo đề gốc)
- Số nút trong khoảng `[1, 10^4]`.
- `1 <= Node.val <= 10^6`.
- `1 <= m, n <= 1000`.

**Follow-up:** sửa danh sách **tại chỗ** (in-place) được không?

Câu hỏi gợi mở:
- Để "xóa" một nút trong linked list, ta có thật sự xóa nó không, hay chỉ **đổi con trỏ** của nút đứng trước?
- Nếu danh sách hết giữa chừng lúc đang giữ hoặc đang xóa thì sao?

## 2. Nền tảng: danh sách liên kết đơn trong Python

### 2.1. Class `ListNode` và hàm dựng / chuyển đổi

Mỗi nút có `val` và `next` (nút kế tiếp hoặc `None`). Ta viết:
- `build_list([1,2,3])` → trả `head` của `1 → 2 → 3`.
- `to_pylist(head)` → trả list Python `[1, 2, 3]` để in và so sánh.

In [1]:
from typing import Optional

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next
    def __repr__(self):
        return f"ListNode({self.val})"

def build_list(values):
    dummy = ListNode()
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def to_pylist(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

def show(head):
    return " -> ".join(map(str, to_pylist(head))) or "(rỗng)"

h = build_list([1, 2, 3])
print(h, h.next, h.next.next, h.next.next.next)
print(to_pylist(h), "|", show(h))

ListNode(1) ListNode(2) ListNode(3) None
[1, 2, 3] | 1 -> 2 -> 3


`build_list` dùng mẹo **nút giả** (`dummy`): một nút đứng trước đầu danh sách giúp mọi nút thật được thêm theo **cùng một cách** (`tail.next = ...`), không cần xử lý riêng nút đầu. Ta sẽ dùng lại mẹo này.

### 2.2. Duyệt danh sách và đếm bước

`while node:` chạy tới khi `node` là `None`. Mỗi bước `node = node.next` — đây là cách **duy nhất** để đi tới (không có chỉ số `head[i]`).

In [2]:
node = build_list([10, 20, 30, 40])
i = 0
while node:
    print(f"vị trí {i}: {node.val}")
    node = node.next
    i += 1

vị trí 0: 10
vị trí 1: 20
vị trí 2: 30
vị trí 3: 40


### 2.3. "Xóa" một nút = nối vòng qua nó

Muốn xóa nút `B` trong `A → B → C`, đứng ở `A` và gán `A.next = B.next` (tức `C`). `B` không còn ai trỏ tới nên coi như đã bị xóa (Python tự thu hồi bộ nhớ).

In [3]:
head = build_list([1, 2, 3])
A = head
print("trước:", show(head))
A.next = A.next.next          # bỏ qua nút 2
print("sau  :", show(head))

trước: 1 -> 2 -> 3
sau  : 1 -> 3


Muốn xóa **một dãy** `n` nút liên tiếp sau `A`: đi một con trỏ `cur` qua `n` nút, rồi gán `A.next = cur` (nút đầu tiên **không** bị xóa).

In [4]:
head = build_list([1, 2, 3, 4, 5])
A = head                       # đứng ở 1, muốn xóa 2, 3, 4
cur = A.next
for _ in range(3):
    cur = cur.next             # sau 3 bước: cur = 5
A.next = cur
print(show(head))              # 1 -> 5

1 -> 5


### 2.4. Toán tử `%` để nhận ra chu kỳ "giữ m, bỏ n"

Đánh số nút từ 0. Quy tắc lặp lại sau mỗi `m + n` nút; trong một chu kỳ, `m` vị trí đầu được giữ. Vậy nút ở vị trí `i` được **giữ** ⇔ `i % (m + n) < m`.

In [5]:
m, n = 2, 3
for i in range(13):
    print(f"i={i:>2} (giá trị {i+1:>2}): i % {m+n} = {i % (m+n)} → {'GIỮ' if i % (m+n) < m else 'xóa'}")

i= 0 (giá trị  1): i % 5 = 0 → GIỮ
i= 1 (giá trị  2): i % 5 = 1 → GIỮ
i= 2 (giá trị  3): i % 5 = 2 → xóa
i= 3 (giá trị  4): i % 5 = 3 → xóa
i= 4 (giá trị  5): i % 5 = 4 → xóa
i= 5 (giá trị  6): i % 5 = 0 → GIỮ
i= 6 (giá trị  7): i % 5 = 1 → GIỮ
i= 7 (giá trị  8): i % 5 = 2 → xóa
i= 8 (giá trị  9): i % 5 = 3 → xóa
i= 9 (giá trị 10): i % 5 = 4 → xóa
i=10 (giá trị 11): i % 5 = 0 → GIỮ
i=11 (giá trị 12): i % 5 = 1 → GIỮ
i=12 (giá trị 13): i % 5 = 2 → xóa


## 3. Cách 1 — Chuyển sang list Python, lọc theo chỉ số, dựng lại

Cách dễ nghĩ nhất cho người quen mảng: đổ linked list ra list, dùng công thức `%` để lọc, rồi dựng danh sách mới.

In [6]:
def delete_nodes_via_list(head, m, n):
    vals = to_pylist(head)
    kept = [v for i, v in enumerate(vals) if i % (m + n) < m]
    return build_list(kept)

print(show(delete_nodes_via_list(build_list(range(1, 14)), 2, 3)))   # 1 -> 2 -> 6 -> 7 -> 11 -> 12
print(show(delete_nodes_via_list(build_list(range(1, 12)), 1, 3)))   # 1 -> 5 -> 9

1 -> 2 -> 6 -> 7 -> 11 -> 12
1 -> 5 -> 9


### 3.1. Vấn đề của cách 1

- Tốn `O(L)` bộ nhớ phụ (list `vals`, `kept`) và tạo **toàn bộ nút mới** — vi phạm follow-up "in-place".
- Trong thực tế, nút có thể mang dữ liệu lớn hoặc được tham chiếu ở chỗ khác; tạo nút mới làm "đứt" những tham chiếu đó.

## 4. Insight — đi theo chu kỳ, chỉ sửa **một** con trỏ mỗi chu kỳ

Trong mỗi chu kỳ:
1. Đi qua `m` nút để giữ; dừng ở **nút giữ cuối cùng** — gọi là `last_kept`.
2. Từ `last_kept.next`, đi qua `n` nút cần xóa; gọi nút ngay sau đó là `after`.
3. Nối `last_kept.next = after` — **một phép gán** xóa cả `n` nút.
4. Tiếp tục chu kỳ mới từ `after`.

Nếu danh sách hết (`None`) ở bất kỳ bước nào thì dừng. Thử tay một chu kỳ trên ví dụ 1:

In [7]:
head = build_list(range(1, 14))
cur = head
for _ in range(2 - 1):          # đi m-1 bước để đứng ở nút giữ thứ m
    cur = cur.next
last_kept = cur
after = last_kept.next
for _ in range(3):              # vượt qua n nút cần xóa
    after = after.next
print("last_kept =", last_kept.val, "| after =", after.val)
last_kept.next = after
print(show(head))

last_kept = 2 | after = 6
1 -> 2 -> 6 -> 7 -> 8 -> 9 -> 10 -> 11 -> 12 -> 13


## 5. Cách 2 — Tạo danh sách mới (không sửa input)

Trung gian giữa cách 1 và in-place: duyệt một lượt, nút nào được giữ thì **tạo bản sao** gắn vào danh sách mới (dùng `dummy` + `tail`). Không cần list Python trung gian, nhưng vẫn tạo nút mới.

In [8]:
def delete_nodes_copy(head, m, n):
    dummy = ListNode()
    tail = dummy
    i = 0
    node = head
    while node:
        if i % (m + n) < m:
            tail.next = ListNode(node.val)
            tail = tail.next
        node = node.next
        i += 1
    return dummy.next

original = build_list(range(1, 14))
result = delete_nodes_copy(original, 2, 3)
print("kết quả:", show(result))
print("input vẫn nguyên:", show(original))

kết quả: 1 -> 2 -> 6 -> 7 -> 11 -> 12
input vẫn nguyên: 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7 -> 8 -> 9 -> 10 -> 11 -> 12 -> 13


## 6. Cách 3 (lời giải chính) — In-place hai pha "giữ m, bỏ n"

### 6.1. Mổ xẻ từng đoạn

```python
cur = head
while cur:
    # Pha GIỮ: đứng ở nút giữ thứ m của chu kỳ (đi m-1 bước)
    for _ in range(m - 1):
        if cur is None: break
        cur = cur.next
    if cur is None:
        break                     # hết danh sách khi đang giữ
    # Pha BỎ: tìm nút ngay sau n nút cần xóa
    after = cur.next
    for _ in range(n):
        if after is None: break
        after = after.next
    cur.next = after              # nối vòng qua n nút
    cur = after                   # chu kỳ mới bắt đầu từ after
return head
```

Điểm tinh tế:
- Pha giữ đi **`m - 1`** bước (không phải `m`), vì `cur` đã đứng sẵn ở nút giữ **thứ nhất**.
- Mọi bước `.next` đều được canh `None` → danh sách hết giữa chừng không gây lỗi.
- `head` không bao giờ bị xóa (vì `m >= 1`), nên trả về `head` luôn đúng.

In [9]:
def delete_nodes_inplace(head, m, n):
    cur = head
    while cur:
        for _ in range(m - 1):
            if cur is None:
                break
            cur = cur.next
        if cur is None:
            break
        after = cur.next
        for _ in range(n):
            if after is None:
                break
            after = after.next
        cur.next = after
        cur = after
    return head

print(show(delete_nodes_inplace(build_list(range(1, 14)), 2, 3)))
print(show(delete_nodes_inplace(build_list(range(1, 12)), 1, 3)))

1 -> 2 -> 6 -> 7 -> 11 -> 12
1 -> 5 -> 9


## 7. TRACE con trỏ — ví dụ 1 (`m = 2`, `n = 3`)

| Chu kỳ | `cur` đầu chu kỳ | `cur` sau pha giữ | `after` sau pha bỏ | gán | danh sách sau |
|---:|---:|---:|---|---|---|
| 1 | 1 | 2 | 6 | `2.next = 6` | 1→2→6→7→…→13 |
| 2 | 6 | 7 | 11 | `7.next = 11` | 1→2→6→7→11→12→13 |
| 3 | 11 | 12 | None (13 rồi hết) | `12.next = None` | 1→2→6→7→11→12 |
| – | None | | | dừng | |

In [10]:
def trace_delete(head, m, n):
    cur, cycle = head, 1
    while cur:
        start = cur.val
        for _ in range(m - 1):
            if cur is None:
                break
            cur = cur.next
        if cur is None:
            print(f"chu kỳ {cycle}: hết danh sách trong pha GIỮ → dừng")
            break
        after = cur.next
        deleted = []
        for _ in range(n):
            if after is None:
                break
            deleted.append(after.val)
            after = after.next
        cur.next = after
        print(f"chu kỳ {cycle}: bắt đầu ở {start}, giữ tới {cur.val}, xóa {deleted}, "
              f"nối {cur.val}.next = {after.val if after else None} | {show(head)}")
        cur = after
        cycle += 1
    return head

trace_delete(build_list(range(1, 14)), 2, 3)

chu kỳ 1: bắt đầu ở 1, giữ tới 2, xóa [3, 4, 5], nối 2.next = 6 | 1 -> 2 -> 6 -> 7 -> 8 -> 9 -> 10 -> 11 -> 12 -> 13
chu kỳ 2: bắt đầu ở 6, giữ tới 7, xóa [8, 9, 10], nối 7.next = 11 | 1 -> 2 -> 6 -> 7 -> 11 -> 12 -> 13
chu kỳ 3: bắt đầu ở 11, giữ tới 12, xóa [13], nối 12.next = None | 1 -> 2 -> 6 -> 7 -> 11 -> 12


ListNode(1)

### 7.1. Trace nhánh "hết danh sách khi đang giữ" và "xóa không đủ n nút"

In [11]:
trace_delete(build_list([1, 2, 3, 4, 5, 6]), 4, 1)
print()
trace_delete(build_list([1, 2, 3]), 1, 5)

chu kỳ 1: bắt đầu ở 1, giữ tới 4, xóa [5], nối 4.next = 6 | 1 -> 2 -> 3 -> 4 -> 6
chu kỳ 2: hết danh sách trong pha GIỮ → dừng

chu kỳ 1: bắt đầu ở 1, giữ tới 1, xóa [2, 3], nối 1.next = None | 1


ListNode(1)

- `[1..6]`, `m=4`, `n=1`: chu kỳ 1 giữ 1–4, xóa 5; chu kỳ 2 bắt đầu ở 6, đi `m-1 = 3` bước thì gặp `None` → dừng, giữ nguyên 6.
- `[1,2,3]`, `m=1`, `n=5`: chỉ còn 2 nút để xóa → `after = None`, `1.next = None`.

## 8. Cách 4 — In-place một vòng với bộ đếm `% (m + n)`

Kết hợp công thức `%` của cách 1 với kỹ thuật nối con trỏ: duyệt từng nút với chỉ số `i`; giữ con trỏ `last` = nút được giữ gần nhất. Nút giữ thì nối `last.next = node`; cuối cùng cắt đuôi `last.next = None`.

In [12]:
def delete_nodes_counter(head, m, n):
    last = None
    node, i = head, 0
    while node:
        nxt = node.next                  # lưu trước vì có thể đổi node.next
        if i % (m + n) < m:
            if last:
                last.next = node
            last = node
        node = nxt
        i += 1
    last.next = None                     # cắt mọi nút bị xóa ở cuối
    return head

print(show(delete_nodes_counter(build_list(range(1, 14)), 2, 3)))
print(show(delete_nodes_counter(build_list(range(1, 12)), 1, 3)))

1 -> 2 -> 6 -> 7 -> 11 -> 12
1 -> 5 -> 9


Vì `head` luôn được giữ (`i = 0`, `0 % (m+n) = 0 < m`), `last` không bao giờ là `None` ở cuối. Cách này đơn giản về logic nhưng thực hiện một phép gán cho **mỗi** nút giữ, còn cách 3 chỉ gán một lần mỗi chu kỳ.

## 9. Cách 5 — Đệ quy

`solve(node)`: giữ `m` nút đầu (đi tới nút giữ thứ `m`), bỏ `n` nút, rồi gắn `solve(phần còn lại)` vào sau. Gọn nhưng độ sâu đệ quy = số chu kỳ ≈ `L / (m+n)`; với `m = n = 1` và `L = 10^4` là 5000 tầng — **vượt** giới hạn mặc định 1000 của Python. Chỉ để tham khảo.

In [13]:
def delete_nodes_recursive(head, m, n):
    if head is None:
        return None
    cur = head
    for _ in range(m - 1):
        if cur.next is None:
            return head                  # hết khi đang giữ
        cur = cur.next
    after = cur.next
    for _ in range(n):
        if after is None:
            break
        after = after.next
    cur.next = delete_nodes_recursive(after, m, n)
    return head

print(show(delete_nodes_recursive(build_list(range(1, 14)), 2, 3)))
print(show(delete_nodes_recursive(build_list(range(1, 12)), 1, 3)))

1 -> 2 -> 6 -> 7 -> 11 -> 12
1 -> 5 -> 9


## 10. Code nộp LeetCode (Python 3)

Trên LeetCode, `ListNode` và `Optional` có sẵn; ở đây đã định nghĩa ở mục 2.

In [14]:
class Solution:
    def deleteNodes(self, head: Optional[ListNode], m: int, n: int) -> Optional[ListNode]:
        cur = head
        while cur:
            for _ in range(m - 1):          # đứng ở nút giữ thứ m
                if cur.next is None:
                    return head
                cur = cur.next
            after = cur.next
            for _ in range(n):              # vượt qua n nút cần xóa
                if after is None:
                    break
                after = after.next
            cur.next = after                # nối vòng
            cur = after
        return head

sol = Solution()
print(show(sol.deleteNodes(build_list(range(1, 14)), 2, 3)))   # 1 -> 2 -> 6 -> 7 -> 11 -> 12
print(show(sol.deleteNodes(build_list(range(1, 12)), 1, 3)))   # 1 -> 5 -> 9

1 -> 2 -> 6 -> 7 -> 11 -> 12
1 -> 5 -> 9


Bản nộp kiểm tra `cur.next is None` **trước** khi bước (thay vì kiểm tra `cur is None` sau khi bước) — gọn hơn: nếu hết danh sách trong pha giữ thì không còn gì để xóa, trả `head` ngay.

## 11. Vì sao thuật toán đúng?

Đánh số nút gốc từ 0. Nút `i` phải được giữ ⇔ `i % (m+n) < m`.

**Invariant** (ở đầu mỗi vòng `while`, với chu kỳ thứ `c` tính từ 0): `cur` là nút gốc ở vị trí `c·(m+n)` (hoặc `None`), và phần danh sách từ `head` tới trước `cur` đã đúng là các nút được giữ của `c` chu kỳ đầu, nối liền theo thứ tự.

- **Khởi tạo** `c = 0`: `cur = head` (vị trí 0), chưa có gì phía trước. ✓
- **Duy trì:** pha giữ đi `m - 1` bước → `cur` ở vị trí `c(m+n) + m - 1` (nút giữ cuối của chu kỳ); các nút ở giữa vẫn nối liền vì ta không sửa gì. Pha bỏ: `after` ở vị trí `c(m+n) + m + n = (c+1)(m+n)`. Gán `cur.next = after` loại đúng `n` nút vị trí `c(m+n)+m .. (c+1)(m+n)-1`. Gán `cur = after` → invariant đúng với `c + 1`. ✓
- **Hết danh sách giữa chừng:** trong pha giữ → mọi nút còn lại đều thuộc phần "giữ", không cần xóa gì → trả `head`. Trong pha bỏ → `after = None`, `cur.next = None` xóa hết phần còn lại (đều thuộc phần "bỏ"). ✓
- **Kết thúc:** vòng lặp dừng khi `cur is None` → mọi chu kỳ đã xử lý. ✓

## 12. Phân tích Time / Space

Gọi `L` = số nút.

| Cách | Time | Space phụ | In-place? |
|---|---|---|---|
| 1. List Python + dựng lại | `O(L)` | `O(L)` | không |
| 2. Tạo danh sách mới | `O(L)` | `O(L)` (nút mới) | không |
| 3. Hai pha giữ/bỏ | `O(L)` | `O(1)` | **có — khuyên dùng** |
| 4. Một vòng + `%` | `O(L)` | `O(1)` | có |
| 5. Đệ quy | `O(L)` | `O(L/(m+n))` call stack | có, nhưng có thể vượt giới hạn đệ quy |

Mỗi nút được con trỏ đi qua đúng một lần → `O(L)`.

## 13. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [15]:
def check(name, head, m, n, expected):
    actual = to_pylist(Solution().deleteNodes(head, m, n))
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

check("ví dụ 1", build_list(range(1, 14)), 2, 3, [1, 2, 6, 7, 11, 12])
check("ví dụ 2", build_list(range(1, 12)), 1, 3, [1, 5, 9])
check("một nút", build_list([7]), 1, 1, [7])
check("m >= độ dài (giữ hết)", build_list([1, 2, 3]), 5, 2, [1, 2, 3])
check("n lớn (xóa hết đuôi)", build_list([1, 2, 3, 4]), 1, 1000, [1])
check("m = n = 1 (giữ chỉ số 0, 2, 4, ...)", build_list(range(1, 8)), 1, 1, [1, 3, 5, 7])
check("dừng đúng lúc hết chu kỳ", build_list(range(1, 11)), 2, 3, [1, 2, 6, 7])
check("hết khi đang giữ", build_list(range(1, 8)), 3, 2, [1, 2, 3, 6, 7])

PASS  ví dụ 1: [1, 2, 6, 7, 11, 12]
PASS  ví dụ 2: [1, 5, 9]
PASS  một nút: [7]
PASS  m >= độ dài (giữ hết): [1, 2, 3]
PASS  n lớn (xóa hết đuôi): [1]
PASS  m = n = 1 (giữ chỉ số 0, 2, 4, ...): [1, 3, 5, 7]
PASS  dừng đúng lúc hết chu kỳ: [1, 2, 6, 7]
PASS  hết khi đang giữ: [1, 2, 3, 6, 7]


### 13.1. Đối chiếu 5 cách — ngẫu nhiên có seed

Các cách in-place **sửa** danh sách, nên mỗi lần gọi ta dựng một danh sách mới từ cùng dữ liệu. Ngoài ra kiểm tra cách 3 thực sự **tái sử dụng nút cũ** (in-place): nút trong kết quả phải là chính các đối tượng ban đầu (`is`).

In [16]:
import random
rng = random.Random(1474)
methods = [delete_nodes_via_list, delete_nodes_copy, delete_nodes_inplace,
           delete_nodes_counter, delete_nodes_recursive, Solution().deleteNodes]
for _ in range(500):
    L = rng.randint(1, 60)
    vals = [rng.randint(1, 10**6) for _ in range(L)]
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    exp = [v for i, v in enumerate(vals) if i % (m + n) < m]
    for f in methods:
        assert to_pylist(f(build_list(vals), m, n)) == exp, (f.__name__, vals, m, n)
    head = build_list(vals)
    originals = []
    node = head
    while node:
        originals.append(node); node = node.next
    res = Solution().deleteNodes(head, m, n)
    node, i = res, 0
    kept_nodes = [nd for i, nd in enumerate(originals) if i % (m + n) < m]
    for nd in kept_nodes:
        assert node is nd
        node = node.next
print("500 bộ ngẫu nhiên: 6 hàm khớp công thức; bản nộp dùng lại đúng các nút cũ (in-place)")

500 bộ ngẫu nhiên: 6 hàm khớp công thức; bản nộp dùng lại đúng các nút cũ (in-place)


### 13.2. Danh sách dài `10^4` nút — bản lặp chạy tốt, bản đệ quy có thể vượt giới hạn

In [17]:
big = list(range(1, 10**4 + 1))
print("bản nộp:", len(to_pylist(Solution().deleteNodes(build_list(big), 1, 1))))
try:
    print("đệ quy:", len(to_pylist(delete_nodes_recursive(build_list(big), 1, 1))))
except RecursionError:
    print("đệ quy: RecursionError (5000 tầng > giới hạn mặc định)")

bản nộp: 5000
đệ quy: RecursionError (5000 tầng > giới hạn mặc định)


## 14. Lỗi người mới hay gặp

### Lỗi 1 — Pha giữ đi `m` bước thay vì `m - 1`

In [18]:
def wrong_m_steps(head, m, n):
    cur = head
    while cur:
        for _ in range(m):                 # đi quá 1 bước
            if cur.next is None:
                return head
            cur = cur.next
        after = cur.next
        for _ in range(n):
            if after is None:
                break
            after = after.next
        cur.next = after
        cur = after
    return head

print("Sai:", to_pylist(wrong_m_steps(build_list(range(1, 14)), 2, 3)), "| Đúng: [1, 2, 6, 7, 11, 12]")

Sai: [1, 2, 3, 7, 8, 9, 13] | Đúng: [1, 2, 6, 7, 11, 12]


**Vì sao sai:** `cur` đã đứng ở nút giữ đầu tiên; đi `m` bước sẽ đứng ở nút **thứ m+1** → giữ thừa 1 nút mỗi chu kỳ.

### Lỗi 2 — Không kiểm tra `None` khi bước → `AttributeError`

In [19]:
def wrong_no_none_check(head, m, n):
    cur = head
    while cur:
        for _ in range(m - 1):
            cur = cur.next
        after = cur.next
        for _ in range(n):
            after = after.next             # after có thể là None
        cur.next = after
        cur = after
    return head

try:
    wrong_no_none_check(build_list([1, 2, 3]), 1, 5)
except AttributeError as e:
    print("Lỗi:", e)

Lỗi: 'NoneType' object has no attribute 'next'


**Sửa:** mỗi lần `.next` phải chắc chắn con trỏ hiện tại khác `None`.

### Lỗi 3 — Quên nối lại con trỏ (chỉ "đi qua" các nút cần xóa)

In [20]:
def wrong_forget_link(head, m, n):
    cur = head
    while cur:
        for _ in range(m - 1):
            if cur.next is None:
                return head
            cur = cur.next
        after = cur.next
        for _ in range(n):
            if after is None:
                break
            after = after.next
        cur = after                        # quên cur.next = after
    return head

print("Sai:", to_pylist(wrong_forget_link(build_list(range(1, 14)), 2, 3)))

Sai: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


**Vì sao sai:** di chuyển con trỏ **không** thay đổi danh sách; chỉ phép gán `.next` mới thay đổi cấu trúc.

### Lỗi 4 — Cách 4 quên cắt đuôi `last.next = None`

In [21]:
def wrong_no_cut(head, m, n):
    last, node, i = None, head, 0
    while node:
        nxt = node.next
        if i % (m + n) < m:
            if last:
                last.next = node
            last = node
        node = nxt
        i += 1
    return head                            # thiếu last.next = None

print("Sai:", to_pylist(wrong_no_cut(build_list(range(1, 14)), 2, 3)), "| Đúng: [1, 2, 6, 7, 11, 12]")

Sai: [1, 2, 6, 7, 11, 12, 13] | Đúng: [1, 2, 6, 7, 11, 12]


**Vì sao sai:** nút giữ cuối (`12`) vẫn trỏ tới `13` — nút lẽ ra bị xóa.

## 15. Mẹo nhận dạng pattern

- "Xóa / bỏ qua nút trong linked list" → đứng ở **nút trước** và gán `prev.next = ...`.
- Quy tắc **lặp theo chu kỳ** (giữ `m`, bỏ `n`; mỗi nút thứ `k`…) → hoặc đếm theo pha, hoặc dùng `i % chu_kỳ`.
- Mọi bước `.next` → hỏi "con trỏ này có thể là `None` không?".
- Khi nút đầu **có thể** bị xóa → dùng **dummy node** (bài này `m >= 1` nên đầu luôn được giữ, nhưng biến thể thì không).
- Bài cùng họ: *Remove Linked List Elements* (203), *Remove Nth Node From End of List* (19), *Delete the Middle Node of a Linked List* (2095), *Odd Even Linked List* (328).

## 16. Biến thể

### 16.1. Remove Linked List Elements (LeetCode 203) — nút đầu có thể bị xóa → dummy

In [22]:
def remove_elements(head, val):
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        if prev.next.val == val:
            prev.next = prev.next.next     # xóa, KHÔNG tiến prev
        else:
            prev = prev.next
    return dummy.next

print(show(remove_elements(build_list([6, 1, 2, 6, 3, 6]), 6)))   # 1 -> 2 -> 3

1 -> 2 -> 3


### 16.2. Xóa mỗi nút thứ `k` (giữ `k-1`, bỏ 1) — chính là bài gốc với `m = k-1`, `n = 1`

In [23]:
def delete_every_kth(head, k):
    if k == 1:
        return None                         # xóa mọi nút
    return Solution().deleteNodes(head, k - 1, 1)

print(show(delete_every_kth(build_list(range(1, 11)), 3)))   # bỏ 3, 6, 9

1 -> 2 -> 4 -> 5 -> 7 -> 8 -> 10


### 16.3. Giữ `m`, bỏ `n` nhưng **trả về danh sách các nút bị xóa**

In [24]:
def split_kept_deleted(head, m, n):
    kept_d, del_d = ListNode(), ListNode()
    kt, dt = kept_d, del_d
    node, i = head, 0
    while node:
        nxt = node.next
        node.next = None
        if i % (m + n) < m:
            kt.next = node; kt = node
        else:
            dt.next = node; dt = node
        node, i = nxt, i + 1
    return kept_d.next, del_d.next

k, d = split_kept_deleted(build_list(range(1, 14)), 2, 3)
print("giữ:", show(k), "| xóa:", show(d))

giữ: 1 -> 2 -> 6 -> 7 -> 11 -> 12 | xóa: 3 -> 4 -> 5 -> 8 -> 9 -> 10 -> 13


## 17. Bài tập luyện tập (tăng dần)

**Bài 1 — Tính tay.** `head = [5,6,7,8,9,10,11]`, `m = 3`, `n = 1`. Liệt kê từng chu kỳ (giữ gì, xóa gì) và kết quả.

**Bài 2 — Dự đoán output.**
```python
h = build_list([1, 2, 3, 4])
h.next = h.next.next.next
print(show(h), h.next.val)
```

Bài 3–6 có cell TODO.

### Bài 3 — TODO: tự viết lời giải in-place hai pha

In [25]:
def exercise_delete_nodes(head, m, n):
    # TODO 1: cur = head; while cur:
    # TODO 2: đi m-1 bước (dừng nếu hết)
    # TODO 3: after = cur.next, vượt n nút (dừng nếu None)
    # TODO 4: cur.next = after; cur = after
    pass

# assert to_pylist(exercise_delete_nodes(build_list(range(1, 14)), 2, 3)) == [1, 2, 6, 7, 11, 12]

### Bài 4 — TODO: Remove Linked List Elements (LeetCode 203) tự viết với dummy

In [26]:
def exercise_remove_elements(head, val):
    # TODO: dummy = ListNode(0, head); prev = dummy; ...
    pass

# assert to_pylist(exercise_remove_elements(build_list([7, 7, 7]), 7)) == []
# assert to_pylist(exercise_remove_elements(build_list([1, 2, 6, 3]), 6)) == [1, 2, 3]

### Bài 5 — TODO: Odd Even Linked List (LeetCode 328)
Gom các nút ở vị trí **lẻ** (1-indexed: 1, 3, 5, …) lên trước, rồi tới các nút vị trí **chẵn**, giữ thứ tự tương đối, in-place `O(1)` bộ nhớ.

In [27]:
def exercise_odd_even(head):
    # TODO: hai con trỏ odd, even; even_head để nối cuối
    pass

# assert to_pylist(exercise_odd_even(build_list([1, 2, 3, 4, 5]))) == [1, 3, 5, 2, 4]
# assert to_pylist(exercise_odd_even(build_list([2, 1, 3, 5, 6, 4, 7]))) == [2, 3, 6, 7, 1, 5, 4]

### Bài 6 — Mini challenge: bỏ `n` rồi giữ `m` (đảo thứ tự hai pha)
Bắt đầu bằng **xóa** `n` nút, rồi **giữ** `m` nút, lặp lại. Nút đầu có thể bị xóa → cần dummy.

In [28]:
def exercise_delete_then_keep(head, m, n):
    # TODO: dummy trỏ tới head; prev = dummy
    #       lặp: vượt n nút từ prev.next, nối prev.next = after; rồi tiến prev m bước
    pass

# assert to_pylist(exercise_delete_then_keep(build_list(range(1, 11)), 2, 3)) == [4, 5, 9, 10]
# assert to_pylist(exercise_delete_then_keep(build_list([1, 2]), 1, 5)) == []

## 18. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 3 | Mỗi chu kỳ chỉ sửa một con trỏ. | `while cur` + hai vòng `for`. | `if cur.next is None: return head` trong pha giữ |
| 4 | Xóa thì không tiến `prev`. | `while prev.next`. | `prev.next = prev.next.next` hoặc `prev = prev.next` |
| 5 | Hai chuỗi xen kẽ, tách rồi nối. | `odd = head; even = even_head = head.next`. | `while even and even.next: odd.next = even.next; odd = odd.next; even.next = odd.next; even = even.next`; cuối `odd.next = even_head` |
| 6 | Giống bài gốc nhưng pha bỏ trước; dummy lo trường hợp đầu bị xóa. | `prev` = nút giữ cuối (ban đầu là dummy). | vượt `n` từ `prev.next` → `after`; `prev.next = after`; đi `m` bước từ `prev` (dừng nếu `None`) |

## 19. Đáp án tham khảo (chỉ xem sau khi tự làm)

**Bài 1:** chu kỳ 1: giữ `5,6,7`, xóa `8`; chu kỳ 2: giữ `9,10,11`, hết → `[5,6,7,9,10,11]`.
**Bài 2:** `1 -> 4 4` (nút 2 và 3 bị bỏ qua).

In [29]:
def answer_delete_nodes(head, m, n):
    cur = head
    while cur:
        for _ in range(m - 1):
            if cur.next is None:
                return head
            cur = cur.next
        after = cur.next
        for _ in range(n):
            if after is None:
                break
            after = after.next
        cur.next = after
        cur = after
    return head

def answer_remove_elements(head, val):
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        if prev.next.val == val:
            prev.next = prev.next.next
        else:
            prev = prev.next
    return dummy.next

def answer_odd_even(head):
    if head is None:
        return None
    odd = head
    even = even_head = head.next
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = even_head
    return head

def answer_delete_then_keep(head, m, n):
    dummy = ListNode(0, head)
    prev = dummy
    while prev:
        after = prev.next
        for _ in range(n):
            if after is None:
                break
            after = after.next
        prev.next = after
        for _ in range(m):
            if prev is None:
                break
            prev = prev.next
    return dummy.next

assert to_pylist(answer_delete_nodes(build_list(range(1, 14)), 2, 3)) == [1, 2, 6, 7, 11, 12]
assert to_pylist(answer_remove_elements(build_list([7, 7, 7]), 7)) == []
assert to_pylist(answer_remove_elements(build_list([1, 2, 6, 3]), 6)) == [1, 2, 3]
assert to_pylist(answer_odd_even(build_list([1, 2, 3, 4, 5]))) == [1, 3, 5, 2, 4]
assert to_pylist(answer_odd_even(build_list([2, 1, 3, 5, 6, 4, 7]))) == [2, 3, 6, 7, 1, 5, 4]
assert to_pylist(answer_delete_then_keep(build_list(range(1, 11)), 2, 3)) == [4, 5, 9, 10]
assert to_pylist(answer_delete_then_keep(build_list([1, 2]), 1, 5)) == []
for _ in range(300):
    vals = [rng.randint(1, 100) for _ in range(rng.randint(1, 40))]
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    exp = [v for i, v in enumerate(vals) if i % (m + n) >= n]
    assert to_pylist(answer_delete_then_keep(build_list(vals), m, n)) == exp
    exp2 = [v for i, v in enumerate(vals) if i % 2 == 0] + [v for i, v in enumerate(vals) if i % 2 == 1]
    assert to_pylist(answer_odd_even(build_list(vals))) == exp2
print("Bài 1:", to_pylist(answer_delete_nodes(build_list([5, 6, 7, 8, 9, 10, 11]), 3, 1)))
h = build_list([1, 2, 3, 4]); h.next = h.next.next.next
print("Bài 2:", show(h), h.next.val)
print("Đáp án bài 3–6: PASS (kể cả đối chiếu ngẫu nhiên)")

Bài 1: [5, 6, 7, 9, 10, 11]
Bài 2: 1 -> 4 4
Đáp án bài 3–6: PASS (kể cả đối chiếu ngẫu nhiên)


## 20. Template nhớ nhanh + checklist submit

```python
cur = head
while cur:
    for _ in range(m - 1):              # GIỮ: đứng ở nút giữ thứ m
        if cur.next is None:
            return head
        cur = cur.next
    after = cur.next
    for _ in range(n):                  # BỎ: vượt n nút
        if after is None:
            break
        after = after.next
    cur.next = after                    # một phép gán xóa cả n nút
    cur = after
return head
```

**Checklist trước khi nộp**
- [ ] Pha giữ đi `m - 1` bước (cur đã đứng ở nút giữ đầu).
- [ ] Mọi `.next` đều được canh `None`.
- [ ] Có phép gán `cur.next = after` (di chuyển con trỏ không làm thay đổi danh sách).
- [ ] Hết danh sách khi đang xóa → `cur.next = None` (cắt đuôi).
- [ ] In-place: không tạo nút mới; trả về `head`.
- [ ] Method tên `deleteNodes(head, m, n)`.